# Session 01 — Weaver's three kinds of problems

This notebook turns a small part of Warren Weaver's **‘Science and Complexity’ (1948)** into a visual experiment. We follow the **same 100 people**, starting in the **same places**, and use one common aggregate measure throughout: **mean pairwise distance**.

> **Guiding question:** When can we understand a population directly, when do statistics help, and when must we also know how people are organized?

No prior Python knowledge is assumed. Run each cell in order with **Runtime → Run all** in Google Colab.

## The three cases

Weaver distinguishes problems involving a few variables, problems involving very many disordered variables that can be handled statistically, and problems whose parts form an **organized whole**. Our miniature model uses these as three ways of arranging movement:

1. **Simplicity:** everyone follows the same known rule.
2. **Disorganized complexity:** each person's action is random and independent; individual paths are unpredictable, but repeated populations show a stable aggregate pattern.
3. **Organized complexity:** actions spread through contacts; outcomes depend on the pattern of relationships.

This is an illustration of Weaver's distinction, not a claim that these toy cases reproduce every detail of his essay.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

plt.style.use('seaborn-v0_8-whitegrid')

N = 100
STEPS = 20
REPLICATIONS = 200
SEED = 7

# 100 people in a 10 by 10 grid. Person 0 starts at (0, 0), etc.
initial_positions = np.array([(x, y) for y in range(10) for x in range(10)], dtype=float)

def mean_pairwise_distance(positions):
    # Distance from every person to every other person
    differences = positions[:, None, :] - positions[None, :, :]
    distances = np.sqrt((differences ** 2).sum(axis=2))
    # Keep each pair once and omit self-distances
    return distances[np.triu_indices(len(positions), k=1)].mean()

initial_mean_distance = mean_pairwise_distance(initial_positions)
print(f'People: {N}')
print(f'Initial mean pairwise distance: {initial_mean_distance:.3f}')

## Case 1 — Simplicity: one rule, directly tractable

Everyone moves exactly **one step right** per time step. We can reason out the result without simulation: translating every point by the same amount cannot change any distance between people. The code checks that reasoning.

In [ ]:
simple_positions = initial_positions.copy()
simple_mean = [mean_pairwise_distance(simple_positions)]

for time in range(STEPS):
    simple_positions[:, 0] += 1  # every x-coordinate increases by one
    simple_mean.append(mean_pairwise_distance(simple_positions))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(initial_positions[:, 0], initial_positions[:, 1], label='start', alpha=.7)
axes[0].scatter(simple_positions[:, 0], simple_positions[:, 1], label='after 20 steps', alpha=.7)
axes[0].set(title='The whole grid is translated', xlabel='x', ylabel='y')
axes[0].axis('equal'); axes[0].legend()
axes[1].plot(range(STEPS + 1), simple_mean, linewidth=3)
axes[1].set(title='Mean pairwise distance is unchanged', xlabel='time', ylabel='mean pairwise distance')
plt.tight_layout(); plt.show()

print(f'Start: {simple_mean[0]:.3f} | End: {simple_mean[-1]:.3f}')

## Case 2 — Disorganized complexity: independent random actions

At each time step, each person independently moves either **0 or 2 steps right**, with equal probability. The expected movement is still one step right per person per time step, so the movement scale is comparable to Case 1.

A particular person's path—and any one run—cannot be predicted exactly. We therefore repeat the model 200 times. The point is **statistical stabilization of the aggregate**, not whether a variable has a normal distribution.

In [ ]:
def run_independent_movement(seed):
    rng = np.random.default_rng(seed)
    positions = initial_positions.copy()
    trajectory = [mean_pairwise_distance(positions)]
    for time in range(STEPS):
        positions[:, 0] += rng.choice([0, 2], size=N)
        trajectory.append(mean_pairwise_distance(positions))
    return trajectory

independent_runs = np.array([run_independent_movement(SEED + r) for r in range(REPLICATIONS)])
independent_average = independent_runs.mean(axis=0)
independent_low = np.percentile(independent_runs, 10, axis=0)
independent_high = np.percentile(independent_runs, 90, axis=0)

plt.figure(figsize=(9, 5))
for run in independent_runs[:12]:
    plt.plot(run, color='gray', alpha=.25)
plt.fill_between(range(STEPS + 1), independent_low, independent_high, alpha=.25, label='middle 80% of runs')
plt.plot(independent_average, linewidth=3, label='average of 200 runs')
plt.xlabel('time'); plt.ylabel('mean pairwise distance')
plt.title('Independent randomness: varied runs, stable aggregate summary')
plt.legend(); plt.show()

summary = pd.DataFrame({
    'time': range(STEPS + 1),
    'average across runs': independent_average,
    '10th percentile': independent_low,
    '90th percentile': independent_high
})
summary.iloc[[0, 5, 10, 15, 20]].round(3)

### Watch the aggregate stabilize

The next plot asks a different question: at the final time, how much does our estimate of the average change as we include more replications? Early estimates jump around. With more runs, the running average settles. This is the statistical tractability Weaver associates with disorganized complexity.

In [ ]:
final_values = independent_runs[:, -1]
running_average = np.cumsum(final_values) / np.arange(1, REPLICATIONS + 1)

plt.figure(figsize=(9, 4))
plt.plot(range(1, REPLICATIONS + 1), running_average, linewidth=2)
plt.axhline(final_values.mean(), color='black', linestyle='--', label='average using all runs')
plt.xlabel('number of replications included')
plt.ylabel('estimated final mean pairwise distance')
plt.title('The aggregate estimate stabilizes with repetition')
plt.legend(); plt.show()

## Case 3 — Organized complexity: actions depend on relationships

Now the same 100 people are nodes in an organizational contact network. In **each replication**, a new random sample of five people initially receives an instruction. A person moves **2 steps right once**, when first receiving it, and then passes it to contacts.

This uses the same two possible actions as Case 2—move 0 or 2—but they are no longer independent. Who moves depends on **who is connected to whom**.

The network has **two disconnected components**, containing 70 and 30 people. Within each component, preferential attachment creates a few highly connected people and many people with fewer contacts. This can represent an informal communication structure that has accumulated around well-connected coordinators or brokers.

The instruction reaches a component only if at least one initially sampled person belongs to it. Even within a reached component, its trajectory depends on whether the initial recipients are central or peripheral. Preferential attachment is a stylized network-growth mechanism—not a complete theory of formal organization.

In [ ]:
# Two separate preferential-attachment components: 70 people and 30 people.
large_component = nx.barabasi_albert_graph(70, m=2, seed=SEED)
small_component = nx.barabasi_albert_graph(30, m=2, seed=SEED + 1)
small_component = nx.relabel_nodes(small_component, {node: node + 70 for node in small_component})
preferential_org = nx.compose(large_component, small_component)

def run_network_movement(network, starting_people):
    positions = initial_positions.copy()
    instructed = set(starting_people)
    frontier = set(starting_people)  # people who act at the next time step
    distances = [mean_pairwise_distance(positions)]
    shares = [len(instructed) / N]

    for time in range(STEPS):
        movers = list(frontier)
        positions[movers, 0] += 2

        # Each recipient transmits once; new recipients act next time step.
        next_frontier = set()
        for person in frontier:
            next_frontier.update(network.neighbors(person))
        frontier = next_frontier - instructed
        instructed.update(frontier)

        distances.append(mean_pairwise_distance(positions))
        shares.append(len(instructed) / N)
    return np.array(distances), np.array(shares)

# Resample the five initial recipients in every replication.
rng = np.random.default_rng(SEED)
organized_runs, organized_shares = [], []
for replication in range(REPLICATIONS):
    starting_people = rng.choice(N, size=5, replace=False)
    distances, shares = run_network_movement(preferential_org, starting_people)
    organized_runs.append(distances); organized_shares.append(shares)

organized_runs = np.array(organized_runs)
organized_shares = np.array(organized_shares)
organized_average = organized_runs.mean(axis=0)

# Node size makes the preferential-attachment hubs visible.
layout = nx.spring_layout(preferential_org, seed=SEED, k=.35, iterations=200)
degrees = dict(preferential_org.degree())
node_sizes = [25 + 10 * degrees[node] for node in preferential_org.nodes()]
node_colors = ['tab:blue' if node < 70 else 'tab:orange' for node in preferential_org.nodes()]
plt.figure(figsize=(11, 7))
nx.draw_networkx(preferential_org, pos=layout, node_size=node_sizes, node_color=node_colors,
                 edge_color='gray', width=.6, alpha=.85, with_labels=False)
plt.title('Preferential-attachment organization: two disconnected components')
plt.axis('off'); plt.show()
print('Contacts:', preferential_org.number_of_edges())
print('Connected components:', nx.number_connected_components(preferential_org))
print('Component sizes:', sorted([len(c) for c in nx.connected_components(preferential_org)], reverse=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for run in organized_shares[:12]:
    axes[0].plot(run, color='gray', alpha=.25)
axes[0].plot(organized_shares.mean(axis=0), linewidth=3, label='average of 200 runs')
axes[0].set(title='Who is sampled affects final reach', xlabel='time', ylabel='share instructed', ylim=(0, 1.05))
axes[0].legend()
for run in organized_runs[:12]:
    axes[1].plot(run, color='gray', alpha=.25)
axes[1].plot(organized_average, linewidth=3, label='average of 200 runs')
axes[1].fill_between(range(STEPS + 1), np.percentile(organized_runs, 10, axis=0),
                     np.percentile(organized_runs, 90, axis=0), alpha=.18)
axes[1].set(title='Relationally dependent movement reaches a plateau', xlabel='time', ylabel='mean pairwise distance')
axes[1].legend()
plt.tight_layout(); plt.show()

## Put the three cases side by side

The lines below all begin with the same people in the same grid and report the same aggregate measure. Each bold stochastic line is an average over 200 replications. In Case 3, five new initial recipients are sampled in every replication. Outcomes depend on which component they enter and where they sit within the preferential-attachment structure.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(simple_mean, linewidth=3, label='1. simplicity: identical movement')
plt.plot(independent_average, linewidth=3, label='2. disorganized: independent average')
plt.fill_between(range(STEPS + 1), independent_low, independent_high, alpha=.12)
plt.plot(organized_average, linewidth=3, label='3. organized: preferential-attachment contacts')
plt.xlabel('time'); plt.ylabel('mean pairwise distance')
plt.title('Same population and measure; different kinds of tractability')
plt.legend(); plt.show()

## Takeaway

| Weaver-inspired case | What happens here? | What makes it tractable? |
|---|---|---|
| **Simplicity** | One known rule translates the whole grid; pairwise distances do not change. | Direct reasoning. |
| **Disorganized complexity** | Individual movements are unpredictable, while repeated runs produce a stable aggregate pattern. | Statistical aggregation. |
| **Organized complexity** | Movement is transmitted relationally; trajectories and final reach depend on where recipients sit in a two-component, hub-centered structure. | Understanding the organized interrelations. |

So the key contrast is not simply ‘few people versus many people.’ It is the kind of dependence among them:

- **Simplicity = directly tractable.**
- **Disorganized complexity = individual unpredictability but statistical aggregate tractability.**
- **Organized complexity = aggregate behavior depends on organized interrelations.**

### Questions for discussion

1. Why is mean pairwise distance unchanged in Case 1 even though everyone moves?
2. What stabilizes in Case 2, and what remains unpredictable?
3. In Case 3, why is knowing the number of initially instructed people insufficient?
4. What might a network link represent in a real organization—communication, authority, advice, or something else?

**Reference:** Weaver, W. (1948). *Science and Complexity*. *American Scientist, 36*(4), 536–544.